In [ ]:
# NORTHSTAR — Tower 1 Masters: Solace Browser QA
# DNA: quality(browser) = design(Ive) × testing(Beck) × security(Bach) × code(Kernighan,Torvalds,Hickey) × infra(Hashimoto) × ux(Norman) × synthesis(DragonRider)
import os, sys, json, re, ast, hashlib, subprocess, importlib
from pathlib import Path

NORTHSTAR = "masters-browser-qa"
NOTEBOOK_PATH = "notebooks/qa/00-masters-browser-qa.ipynb"
TOWER = 1
TOWER_NAME = "Tower of Masters"
TOTAL_PROBES = 20
MIN_SCORE = 70

# Project root
BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

# Results tracking
results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" — {detail}" if detail else ""))

print(f"Tower {TOWER}: {TOWER_NAME}")
print(f"Target: solace-browser")
print(f"Probes: {TOTAL_PROBES}")
print(f"Browser root: {BROWSER_ROOT}")
assert BROWSER_ROOT.exists(), f"Browser root not found: {BROWSER_ROOT}"


In [ ]:
# T1 F1 — Jony Ive (Design)
# Q: Can a keyboard-only user navigate all tabs without mouse?
# Q: Is there a visible focus indicator on every interactive element?
# Q: Are there more than 5 font sizes on any single page?

css_path = WEB / "css" / "site.css"
assert css_path.exists(), f"CSS file not found: {css_path}"
css_content = css_path.read_text()

# Check 1: Focus indicators exist in CSS
focus_rules = re.findall(r':focus[^{]*\{[^}]+\}', css_content, re.DOTALL)
has_focus = len(focus_rules) > 0
record("F1-P1", "Focus indicators in CSS", has_focus, 
       f"{len(focus_rules)} :focus rules found")

# Check 2: Skip-nav link exists (via layout.js dynamic injection or static HTML)
layout_js = WEB / "js" / "layout.js"
skip_nav_in_js = False
if layout_js.exists():
    layout_content = layout_js.read_text()
    skip_nav_in_js = bool(re.search(r'skip.nav|skip.content|skiplink', layout_content, re.IGNORECASE))
# Also check static HTML as fallback
html_files = list(WEB.glob("*.html"))
skip_nav_in_html = sum(1 for hf in html_files 
    if re.search(r'skip.*nav|skip.*content|skiplink', hf.read_text(), re.IGNORECASE))
has_skip_nav = skip_nav_in_js or skip_nav_in_html > 0
record("F1-P2", "Skip navigation links exist", has_skip_nav,
       f"layout.js injection={skip_nav_in_js}, static HTML={skip_nav_in_html}/{len(html_files)}")

# Check 3: Font size discipline (count unique font-size declarations)
font_sizes = re.findall(r'font-size:\s*([^;]+)', css_content)
unique_sizes = set(s.strip() for s in font_sizes)
# More than 12 unique sizes = typographic chaos
size_ok = len(unique_sizes) <= 12
record("F1-P3", "Font size discipline (<= 12 unique sizes)", size_ok,
       f"{len(unique_sizes)} unique sizes: {sorted(unique_sizes)[:8]}")


In [ ]:
# T1 F2 — Kent Beck (Testing)
# Q: Is there a test for every public API endpoint?
# Q: Does the test suite exist and have meaningful coverage?

test_files = list(TESTS.glob("test_*.py"))
record("F2-P1", "Test files exist", len(test_files) > 0,
       f"{len(test_files)} test files found")

# Check which key modules have corresponding tests
key_modules = ["auth_proxy", "budget_gates", "capture_pipeline", "session_manager",
               "consent_ui", "evidence", "app_store", "channels", "canvas"]
tested_modules = []
untested_modules = []
for mod in key_modules:
    test_path = TESTS / f"test_{mod}.py"
    has_test = any(mod in tf.name for tf in test_files)
    if has_test:
        tested_modules.append(mod)
    else:
        untested_modules.append(mod)

coverage = len(tested_modules) / len(key_modules) * 100
record("F2-P2", "Key modules have tests", coverage >= 70,
       f"{len(tested_modules)}/{len(key_modules)} ({coverage:.0f}%): untested={untested_modules}")

# Check for assertion density (tests should have assert statements)
total_asserts = 0
for tf in test_files[:10]:  # sample first 10
    content = tf.read_text()
    asserts = len(re.findall(r'\bassert\b', content))
    total_asserts += asserts
avg_asserts = total_asserts / min(len(test_files), 10) if test_files else 0
record("F2-P3", "Tests have adequate assertions", avg_asserts >= 3,
       f"Avg {avg_asserts:.1f} assertions per test file (sample of {min(len(test_files), 10)})")


In [ ]:
# T1 F3 — James Bach (Security)
# Q: Does OAuth3 token validation fail-closed?
# Q: Can an attacker inject into the evidence chain?
# Q: Does vault decryption fail loudly on corrupted data?

# Check 1: OAuth3 enforcement exists and is fail-closed
enforcement_path = SRC / "oauth3" / "enforcement.py"
if enforcement_path.exists():
    enforcement_code = enforcement_path.read_text()
    # Look for fail-closed patterns (raise on invalid, not return default)
    has_raise = "raise" in enforcement_code
    has_return_none = re.search(r'return\s+None', enforcement_code)
    fail_closed = has_raise and not has_return_none
    record("F3-P1", "OAuth3 enforcement is fail-closed", fail_closed,
           f"has raise={has_raise}, return None={bool(has_return_none)}")
else:
    record("F3-P1", "OAuth3 enforcement module exists", False, "enforcement.py not found")

# Check 2: Evidence chain uses hash chaining
evidence_files = list((SRC / "evidence").glob("*.py")) if (SRC / "evidence").exists() else []
has_hash_chain = False
for ef in evidence_files:
    content = ef.read_text()
    if "sha256" in content.lower() or "hashlib" in content:
        has_hash_chain = True
        break
if not evidence_files:
    # Check capture_pipeline
    cap = SRC / "capture_pipeline.py"
    if cap.exists():
        content = cap.read_text()
        has_hash_chain = "sha256" in content.lower() or "hashlib" in content
record("F3-P2", "Evidence uses cryptographic hashing", has_hash_chain)

# Check 3: No bare except blocks in security-critical code
security_files = list((SRC / "oauth3").glob("*.py"))
bare_excepts = 0
for sf in security_files:
    content = sf.read_text()
    bare_excepts += len(re.findall(r'except\s*:', content))
    bare_excepts += len(re.findall(r'except\s+Exception\s*:', content))
record("F3-P3", "No bare excepts in OAuth3 code", bare_excepts == 0,
       f"{bare_excepts} bare except blocks found")

# Check 4: Vault uses proper key derivation
vault_path = SRC / "oauth3" / "vault.py"
if vault_path.exists():
    vault_code = vault_path.read_text()
    has_kdf = any(kw in vault_code.lower() for kw in ["pbkdf2", "argon2", "scrypt", "key_derivation"])
    has_aes = "aes" in vault_code.lower() or "gcm" in vault_code.lower()
    record("F3-P4", "Vault uses proper key derivation", has_kdf,
           f"KDF={has_kdf}, AES={has_aes}")
else:
    record("F3-P4", "Vault module exists", False, "vault.py not found")


In [ ]:
# T1 F7 — Kernighan (Clarity) + F8 — Torvalds (Correctness) + F9 — Hickey (Simplicity)
# Q: Is there innerHTML assignment with user-controlled data?
# Q: Are there race conditions in concurrent code?
# Q: Does the system use immutable data where possible?
# Q: Does the code fail loudly on missing config?

# Check 1: No dangerous innerHTML in JS — only flag user-data interpolations WITHOUT escapeHtml
js_path = WEB / "js" / "solace.js"
if js_path.exists():
    js_content = js_path.read_text()
    all_innerhtml = re.findall(r'\.innerHTML\s*=(?!=)', js_content)
    # Split by innerHTML and check each for unescaped user data
    parts = re.split(r'\.innerHTML\s*=', js_content)
    unsafe_count = 0
    for part in parts[1:]:
        chunk = part[:500]
        interps = re.findall(r'\$\{([^}]+)\}', chunk)
        for interp in interps:
            interp = interp.strip()
            if 'escapeHtml' in interp:
                continue  # already escaped
            if re.match(r'^[\d\s+\-*/.]+$', interp):
                continue  # numeric expression
            if re.match(r'^"[^"]*"$', interp) or re.match(r"^'[^']*'$", interp):
                continue  # static string
            # Safe patterns: render functions that escape internally, class helpers
            if any(fn in interp for fn in ['render', 'statusTag', 'statusPill', 'index', '|| "?"', '|| "', "|| '"]):
                continue
            unsafe_count += 1
    record("F7-P1", "No unsafe innerHTML in solace.js", unsafe_count == 0,
           f"{len(all_innerhtml)} total innerHTML, {unsafe_count} truly unsafe")
else:
    record("F7-P1", "solace.js exists", False)

# Check 2: No os.getenv with mock fallback in Python code (Fallback Ban)
py_files = list(SRC.rglob("*.py"))
mock_fallbacks = 0
mock_examples = []
for pf in py_files[:50]:  # sample 50 files
    try:
        content = pf.read_text()
        # Look for os.getenv("KEY", "some_default") where default looks like a mock
        matches = re.findall(r'os\.getenv\([^)]*,\s*["\'](mock|fake|test|dummy|default|placeholder)[^"]*["\']]', content, re.IGNORECASE)
        if matches:
            mock_fallbacks += len(matches)
            mock_examples.append(str(pf.relative_to(BROWSER_ROOT)))
    except Exception:
        pass
record("F8-P1", "No mock credential fallbacks (Fallback Ban)", mock_fallbacks == 0,
       f"{mock_fallbacks} mock fallbacks in {mock_examples}" if mock_fallbacks else "clean")

# Check 3: Server has proper error handling (no catch-and-ignore in real code, not docstrings)
server_path = BROWSER_ROOT / "solace_browser_server.py"
if server_path.exists():
    server_code = server_path.read_text()
    # Find except...pass that's NOT inside a docstring or comment
    lines = server_code.split('\n')
    catch_ignore = 0
    in_docstring = False
    for i, line in enumerate(lines):
        stripped = line.strip()
        # Track docstring boundaries
        if '"""' in stripped or "'''" in stripped:
            count = stripped.count('"""') + stripped.count("'''")
            if count % 2 == 1:
                in_docstring = not in_docstring
        if in_docstring or stripped.startswith('#'):
            continue
        if re.match(r'except[^:]*:\s*$', stripped):
            # Check if next non-empty line is 'pass'
            for j in range(i+1, min(i+3, len(lines))):
                next_stripped = lines[j].strip()
                if next_stripped == 'pass':
                    catch_ignore += 1
                    break
                elif next_stripped and not next_stripped.startswith('#'):
                    break
    record("F8-P2", "No catch-and-ignore in browser server", catch_ignore == 0,
           f"{catch_ignore} except...pass blocks (excluding docstrings/comments)")
else:
    record("F8-P2", "Browser server exists", False)

# Check 4: Configuration validated at startup (not silently defaulted)
if server_path.exists():
    server_code = server_path.read_text()
    has_startup_validation = bool(re.search(r'(argparse|parser\.add_argument|if not .*config)', server_code))
    record("F9-P1", "Server validates config at startup", has_startup_validation)

In [ ]:
# T1 F14 — Mitchell Hashimoto (Security Infrastructure)
# Q: Can an attacker learn internal file structure from API errors?
# Q: Does the vault use proper key derivation?
# Q: Are audit log writes using fsync?
# Q: Is CORS restrictive?

# Check 1: CORS configuration
server_path = BROWSER_ROOT / "solace_browser_server.py"
if server_path.exists():
    server_code = server_path.read_text()
    wildcard_cors = "'*'" in server_code or '"*"' in server_code
    has_cors = bool(re.search(r'cors|access.control', server_code, re.IGNORECASE))
    record("F14-P1", "CORS is not wildcard", not wildcard_cors,
           f"CORS present={has_cors}, wildcard={wildcard_cors}")

# Check 2: Error responses don't leak file paths
    error_handlers = re.findall(r'(traceback|stack_trace|__file__|os\.path\.abspath).*response', server_code, re.IGNORECASE)
    record("F14-P2", "Error responses don't leak internals", len(error_handlers) == 0,
           f"{len(error_handlers)} potential info leaks")

# Check 3: Token stored in memory only (not disk) in browser context
token_path = SRC / "oauth3" / "token.py"
if token_path.exists():
    token_code = token_path.read_text()
    disk_writes = re.findall(r'(open\(|write\(|json\.dump|pickle|shelve)', token_code)
    memory_only = len(disk_writes) == 0
    record("F14-P3", "Tokens are memory-only (not written to disk)", memory_only,
           f"{'No' if memory_only else len(disk_writes)} disk write calls in token.py")
else:
    record("F14-P3", "Token module exists", False)

# Check 4: Step-up auth exists
step_up_path = SRC / "oauth3" / "step_up.py"
step_up_exists = step_up_path.exists() and step_up_path.stat().st_size > 100
record("F14-P4", "Step-up authentication module exists", step_up_exists)


In [ ]:
# T1 F17 — Don Norman (UX Design)
# Q: Does every form control have an explicit label?
# Q: Are there false affordances (things that look clickable but are not)?
# Q: Is there a visible signifier for every affordance?

html_files = list(WEB.glob("*.html"))
total_inputs = 0
labeled_inputs = 0
pages_with_forms = 0

for hf in html_files:
    content = hf.read_text()
    inputs = re.findall(r'<input[^>]*>', content, re.IGNORECASE)
    selects = re.findall(r'<select[^>]*>', content, re.IGNORECASE)
    textareas = re.findall(r'<textarea[^>]*>', content, re.IGNORECASE)
    all_controls = inputs + selects + textareas
    
    if all_controls:
        pages_with_forms += 1
    
    for ctrl in all_controls:
        total_inputs += 1
        # Check if control has id/name that matches a label
        ctrl_id = re.search(r'id=["\'](\w+)', ctrl)
        has_aria = bool(re.search(r'aria-label', ctrl))
        has_placeholder = bool(re.search(r'placeholder=', ctrl))
        is_hidden = bool(re.search(r'type=["\'](hidden|submit)', ctrl))
        
        if is_hidden or has_aria or has_placeholder:
            labeled_inputs += 1
        elif ctrl_id:
            # Check if there is a <label for="id"> in the same file
            label_match = re.search(f'<label[^>]*for=["\']{ctrl_id.group(1)}', content)
            if label_match:
                labeled_inputs += 1

label_rate = (labeled_inputs / total_inputs * 100) if total_inputs > 0 else 100
record("F17-P1", "Form controls have labels/aria", label_rate >= 80,
       f"{labeled_inputs}/{total_inputs} ({label_rate:.0f}%) across {pages_with_forms} pages with forms")

# Check 2: Buttons have text or aria-label
total_buttons = 0
labeled_buttons = 0
for hf in html_files:
    content = hf.read_text()
    buttons = re.findall(r'<button[^>]*>([^<]*)</button>', content, re.IGNORECASE | re.DOTALL)
    btn_tags = re.findall(r'<button[^>]*>', content, re.IGNORECASE)
    for i, btn_tag in enumerate(btn_tags):
        total_buttons += 1
        has_text = i < len(buttons) and buttons[i].strip()
        has_aria = bool(re.search(r'aria-label', btn_tag))
        if has_text or has_aria:
            labeled_buttons += 1

btn_rate = (labeled_buttons / total_buttons * 100) if total_buttons > 0 else 100
record("F17-P2", "Buttons have visible text or aria-label", btn_rate >= 80,
       f"{labeled_buttons}/{total_buttons} ({btn_rate:.0f}%)")


In [ ]:
# T1 F46 — Satoshi Nakamoto (Trustless Verification)
# Q: Does the evidence system produce tamper-evident records?
# Q: Is there a hash chain for audit events?
# Q: Can evidence be verified without trusting the source?

# Check evidence module
evidence_dir = SRC / "evidence"
if evidence_dir.exists():
    evidence_files = list(evidence_dir.glob("*.py"))
    evidence_code = ""
    for ef in evidence_files:
        evidence_code += ef.read_text()
    
    has_hash = "sha256" in evidence_code or "hashlib" in evidence_code
    has_chain = "chain" in evidence_code.lower() or "previous" in evidence_code.lower()
    has_timestamp = "timestamp" in evidence_code.lower() or "datetime" in evidence_code.lower()
    has_immutable = "append" in evidence_code.lower() and "only" in evidence_code.lower()
    
    record("F46-P1", "Evidence uses SHA-256 hashing", has_hash)
    record("F46-P2", "Evidence implements hash chaining", has_chain)
    record("F46-P3", "Evidence includes timestamps", has_timestamp)
else:
    # Check capture_pipeline as alternative
    cap = SRC / "capture_pipeline.py"
    if cap.exists():
        cap_code = cap.read_text()
        has_hash = "sha256" in cap_code or "hashlib" in cap_code
        has_chain = "chain" in cap_code.lower()
        record("F46-P1", "Evidence/capture uses hashing", has_hash)
        record("F46-P2", "Evidence implements chaining", has_chain)
        record("F46-P3", "Evidence module exists", True, "via capture_pipeline.py")
    else:
        record("F46-P1", "Evidence module exists", False, "no evidence/ dir or capture_pipeline.py")


In [ ]:
# T1 F48 — Dragon Rider (Integration) + F49 — Bruce Lee (Jeet Kune Do)
# Q: Does the integration produce a coherent quality picture?
# Q: Is the product's greatest strength also its greatest vulnerability?

# Overall architecture coherence check
key_components = {
    "oauth3": SRC / "oauth3",
    "evidence": SRC / "evidence" if (SRC / "evidence").exists() else SRC / "capture_pipeline.py",
    "budget_gates": SRC / "budget_gates.py",
    "browser_server": BROWSER_ROOT / "solace_browser_server.py",
    "recipes": SRC / "recipes",
    "session_manager": SRC / "session_manager.py",
    "settings": SRC / "settings_manager.py",
    "web_css": WEB / "css" / "site.css",
    "web_js": WEB / "js" / "solace.js",
    "tests": TESTS,
}

present = 0
missing = []
for name, path in key_components.items():
    if path.exists():
        present += 1
    else:
        missing.append(name)

coherence = present / len(key_components) * 100
record("F48-P1", "Core architecture components present", coherence >= 80,
       f"{present}/{len(key_components)} ({coherence:.0f}%)" + (f" missing: {missing}" if missing else ""))

# Check NORTHSTAR alignment
northstar_path = BROWSER_ROOT / "NORTHSTAR.md"
claude_path = BROWSER_ROOT / "CLAUDE.md"
northstar_exists = northstar_path.exists()
claude_exists = claude_path.exists()
record("F48-P2", "NORTHSTAR.md and CLAUDE.md exist", northstar_exists and claude_exists)

# F49 Bruce Lee: "Be water" — check if the system is adaptable
# Proxy: does the codebase have configuration/settings that allow runtime adaptation?
settings_path = SRC / "settings_manager.py"
if settings_path.exists():
    settings_code = settings_path.read_text()
    has_hot_reload = "reload" in settings_code.lower() or "watch" in settings_code.lower()
    record("F49-P1", "Settings support runtime adaptation", has_hot_reload,
           "hot-reload capability")
else:
    record("F49-P1", "Settings manager exists", False)


In [ ]:
# EVIDENCE SUMMARY — Tower 1 Masters × Solace Browser
import json
from datetime import datetime

passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
finding_rate = (failed / total * 100) if total > 0 else 0

summary = {
    "surface": NORTHSTAR,
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total,
    "passed": passed,
    "failed": failed,
    "finding_rate": round(finding_rate, 1),
    "score": round(passed / total * 100, 1) if total > 0 else 0,
    "min_score": MIN_SCORE,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME} — SOLACE BROWSER")
print("=" * 60)
print(f"Total probes:  {total}")
print(f"Passed:        {passed}")
print(f"Failed:        {failed}")
print(f"Score:         {summary['score']}%")
print(f"Finding rate:  {finding_rate:.1f}%")
print(f"Converged:     {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
print()
if summary["findings"]:
    print("FINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))
